# Chapter 3 - Lab 6: <font color='blue'>LlamaIndex Agent</font>

**<font color='purple'>Goal</font>**:
In this lab, you will build a **financial analysis agent using LlamaIndex** that compares the P/E (Price/Earnings) ratios of two companies — Apple (AAPL) and JPMorgan (JPM) — and produces a short investment memo.

LlamaIndex is best known as a RAG / indexing framework, but it ships a clean agent abstraction — `FunctionAgent` — that is async-first and integrates with the same data connectors you would use for retrieval pipelines. In this lab you will use it to solve the standard P/E comparison task and notice how compact the agent definition is.

This is the same reference task used across every framework lab in Chapter 3 — comparing all of them on the *same* task makes the differences in API style, abstractions, and ergonomics easy to spot.

**<font color='purple'>Tech stack</font>**:

* **LlamaIndex** (`llama-index`, `llama-index-llms-openai`) — `FunctionAgent` from `core.agent.workflow`.
* **OpenAI** `gpt-4o-mini` (via `llama_index.llms.openai.OpenAI`).
* **Python `asyncio`** — `FunctionAgent.run` is async.

You will need an OpenAI API key with some credits available.

## 1. Install packages

Install the framework and its dependencies.

In [ ]:
%pip install -q llama-index llama-index-llms-openai pydantic python-dotenv

## 2. Set up the API key(s)

This lab needs the following key(s):

  * **`OPENAI_API_KEY`** — your OpenAI key

If you are running this notebook in **Google Colab**, add each key in the left vertical menu under the *key* icon, using the names above.

If you are running locally, set the same names as environment variables (or load them from a `.env` file).

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally — assume env vars are already set (e.g. via .env).
    pass

## 3. Bootstrap the shared task setup

Every framework lab in this chapter shares the same task, tools, finance dataset, and prompts. These are factored out into `common.py`. If you have cloned the book's repository, `common.py` is already on disk; otherwise the cell below downloads it for you.

In [ ]:
import os, urllib.request

if not os.path.exists('common.py'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/PacktPublishing/Building-AI-Agents-for-Finance-/main/Chapter%203/common.py',
        'common.py',
    )

from common import (
    get_stock_data,
    compute_pe,
    finance_data,
    system_message,
    input_message,
)

print('Tools loaded. Reference task:')
print(' ', input_message)

## 4. Build the LlamaIndex agent

Pass the LLM, the tools, and the system prompt to `FunctionAgent`. There is no intermediate `Tool` wrapper — plain Python functions are accepted directly, with the schema inferred from type hints.

In [ ]:
import os
from llama_index.llms.openai import OpenAI
from llama_index.core.agent.workflow import FunctionAgent

llm = OpenAI(
    model='gpt-4o-mini',
    api_key=os.environ['OPENAI_API_KEY'],
    temperature=0,
)

agent = FunctionAgent(
    tools=[get_stock_data, compute_pe],
    llm=llm,
    system_prompt=system_message,
)

## 5. Run the agent

`agent.run` is asynchronous and returns a response object that knows how to render itself as a string. In a notebook we can simply `await` the call.

In [ ]:
response = await agent.run(input_message)
print(str(response))

## 6. Results

You should see the agent call `get_stock_data` (once per ticker), then call `compute_pe` to compute each P/E ratio, and finally produce a short memo comparing the two.

**What to notice about LlamaIndex specifically:**

* The smallest *agent construction* code in this chapter — one `FunctionAgent` call.
* The same library that powers your RAG indexing also runs the agent loop, which is convenient when you eventually want the agent to consult a knowledge base.
* Output is a single response object rather than a message trace — for step-by-step inspection you would use LlamaIndex's `AgentWorkflow` (which Chapter 7 explores).
* Trade-off: less ceremony, but also less visibility into intermediate reasoning compared to AutoGen or ADK.